# Lab 1, part A: the tools

Lab 1 (A+B) costs: $0.099

**Part A builds an MCP server over three knowledge sources, gives its failures a shape an
agent can act on, and wires it into Claude Code.**

Scenario 4, developer productivity. Fernhill keeps what its engineers need to know in three
places: a ticket queue, a service catalogue, and a set of architecture decisions.

**No API calls here.** Everything runs offline, and the proof that it works is a terminal
session at the end rather than a Python call. Part B, after Chapter 7, hands the same server
to an Agent SDK consumer.

## 1. A workspace to serve

Nothing in `fixtures.py` is Claude specific. It is the material the tools will expose: three
tickets, a service catalogue, two architecture decision records, and three small Python
files.

**The Python files carry two deliberate traps**, and part B collects on both. One function is
re-exported under two other names, so searching for the original name finds a third of the
callers. And one line appears twice, identically, which is what makes an edit anchor
ambiguous.

The servers are checked in beside this notebook and **staged into `workspace/`**, which is
the sandbox: gitignored, what the terminal steps below `cd` into, and safe to delete and
rebuild at any point.

In [ ]:
import json

import labkit

lab = labkit.start(model_env=None, credential="none")

import fixtures

fixtures.build(lab.workspace)
SERVERS = lab.stage("servers")[0]      # checked-in source -> workspace/servers/
print(f"servers staged at {SERVERS.relative_to(lab.dir)}")

## 2. What a tool returns when it fails

**A tool has two jobs. Return an answer, and, when it cannot, return a failure the agent can
act on. The second one is the one that gets skipped.**

`isError` on the result says only that something broke. On its own it leaves the agent
guessing between four different recoveries, two of which are expensive: retry a policy wall
forever, or abandon a timeout that would have worked on the next attempt.

So the payload carries the category, whether a retry can help, and what to do instead.

| Category | Retryable | What the agent should do |
|---|---|---|
| transient | yes | retry, with a delay |
| validation | yes, but only after the input changes | fix the arguments, then retry |
| business | no | explain it to the person, in their terms |
| permission | no | escalate |

**And the case that is not a failure at all:** a query that ran correctly and matched nothing
is a success with an empty list. Mark it as an error and the agent retries a question that
was already answered.

One thing to notice in `errors.py`: the helper builds an **exception**. With FastMCP,
raising is what sets `isError` on the result. A returned dictionary that happens to describe
a failure is reported as a success. Lab 3's Agent SDK tools do the opposite and return a flag.

In [ ]:
labkit.show_source(SERVERS / "errors.py", "tool_error")

## 3. The five shapes, side by side

Four failures and one success. **The fifth row is the one worth pausing on: it is not an
error, and the agent must not treat it as one.**

In [ ]:
from errors import tool_error

CASES = [
    tool_error("transient", "The ticket store did not respond in time.",
               "Retry in a few seconds."),
    tool_error("validation", "No service called 'refund'. Expected one of: refunds, billing, checkout.",
               "Call again with one of the listed names."),
    tool_error("business", "That decision record is superseded and is not served any more.",
               "Tell the engineer it was superseded by ADR-0005, and read that instead."),
    tool_error("permission", "The ticket queue is readable only by the Payments group.",
               "Ask a Payments engineer, or escalate."),
]

rows = []
for case in CASES:
    payload = json.loads(str(case))
    rows.append((payload["errorCategory"], str(payload["isRetryable"]), payload["nextAction"]))
rows.append(("(no error)", "n/a",
             json.dumps({"query": "kubernetes", "matched": 0, "results": []})))

labkit.show_table(rows, headers=("category", "retry?", "next action"),
                  wrap={"next action": 58})

## 4. The server

**`FastMCP` turns decorated functions into tools, and `Field(description=...)` on each
argument is what gives Claude a per-argument description in the generated schema.**

Two things to notice below, because part B is going to collect on both.

**The descriptions are thin**, and the first tool is worse than thin: it is called `lookup`
and it says `Look things up.` Nothing there tells anyone, human or model, that this is the
support ticket queue. That is not a style problem, it is the selection mechanism, and part B
measures exactly what it costs.

**And `architecture_decisions` is doing two jobs**: listing what exists, and fetching one. A
tool that does two things is harder to describe than either of them separately.

In [ ]:
labkit.show_source(SERVERS / "devtools_server.py", "lookup", "architecture_decisions")

## 5. Call the tools directly

**No model involved.** A decorated FastMCP tool is still an ordinary function, so the fastest
way to know the server works is to call it. Three outcomes below: a hit, a query that matched
nothing, and a failure that raises.

**One wrinkle worth knowing**, because it will bite you the first time you improvise here.
Pass every argument explicitly. An argument you leave out does not fall back to its default:
it arrives as the `Field` descriptor itself, and the tool fails on a type nobody expected.

In [ ]:
import devtools_server
from devtools_server import architecture_decisions, lookup, service_catalogue
from errors import ToolError

hit = lookup(query="duplicate charges")
print(f"hit         matched {hit['matched']}: {hit['results'][0]['id']} {hit['results'][0]['title']}")

nothing = lookup(query="kubernetes")
print(f"no matches  matched {nothing['matched']}, results {nothing['results']}, and nothing raised")

print(f"catalogue   {service_catalogue(name='refunds')}")
print(f"decisions   {architecture_decisions(adr_id='')['available']}")

try:
    service_catalogue(name="refund")
except ToolError as exc:
    print(f"failure     {exc}")

## 6. What the model actually sees

**This is the whole basis on which Claude chooses. Not the code, not the data: the name, the
description, and the argument descriptions.**

Read the three below as if you were the one choosing. Given a repository you can also search
with `Grep`, is there anything here that would make you pick these?

In [ ]:
labkit.show_tools(await devtools_server.mcp.list_tools())

## 7. Two scopes, both loaded at once

**Claude Code reads server definitions from two places, and loads both at the same time.**

**Project scope** is `.mcp.json` at the project root, in version control, for the servers the
whole team needs. Because it is committed, a credential never goes in it literally: write
`${DEVTOOLS_TOKEN}` and each engineer supplies their own from the environment.
`${VAR:-default}` falls back when the variable is unset.

**User scope** is your own file, for personal and experimental servers, added with
`claude mcp add --scope user`. It follows you between projects and nobody else sees it.

Once connected, the tools arrive named `mcp__<server>__<tool>`, so the server this cell
registers gives Claude `mcp__devtools__lookup`, which is exactly the name part B is going to
take apart.

In [ ]:
config = {
    "mcpServers": {
        "devtools": {
            "command": "uv",
            "args": ["run", "--project", "../..", "python", "servers/devtools_server.py"],
            "env": {"DEVTOOLS_TOKEN": "${DEVTOOLS_TOKEN:-local-dev}"},
        }
    }
}

(lab.workspace / ".mcp.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
print(json.dumps(config, indent=2))

## 8. Now prove it, in Claude Code

**The notebook has not spoken to a model once. This is where that happens, and Claude Code is
the client.**

Open a terminal in the workspace directory this notebook just created:

```bash
cd workspace
claude
```

**One.** Run `/mcp`. The `devtools` server should be listed and connected. That is project
scope, from the `.mcp.json` written above.

**Two.** Ask it something the server can answer:

> which open tickets mention duplicate charges?

Watch which tool it reaches for, and write it down. There is a `Grep` available to it and a
`lookup` described as "Look things up." Whichever wins, **that observation is what part B
starts from**.

**Three.** Add the same server again under user scope, to see both at once:

```bash
claude mcp add --scope user devtools-personal -- uv run --project ../.. python servers/devtools_server.py
```

Restart `claude`, run `/mcp`, and both entries are listed: one from the project file, one
from your own. Then clean up, because this one follows you out of the project:

```bash
claude mcp remove --scope user devtools-personal
```

Stop here until after Chapter 7.